In [3]:
import duckdb
import time

def ingest_csv_to_parquet(input_filepath, output_filepath):
    print(f"Starting ingestion process...")
    print(f"Input CSV: {input_filepath}")
    print(f"Output Parquet: {output_filepath}")
    
    start_time = time.time()
    
    # Connect to DuckDB (using a file-backed DB in case of memory spilling for 35GB)
    con = duckdb.connect('duckdb_ingestion_temp.db')
    
    # DuckDB's COPY command efficiently streams data directly from CSV to Parquet 
    # out-of-core, without loading everything into RAM at once.
    query = f"""
        COPY (
            SELECT * 
            FROM read_csv_auto('{input_filepath}', sample_size=-1)
        ) TO '{output_filepath}' (FORMAT 'PARQUET', CODEC 'SNAPPY');
    """
    
    try:
        con.execute(query)
        end_time = time.time()
        print(f"Success! Conversion completed in {(end_time - start_time):.2f} seconds.")
    except Exception as e:
        print(f"Error during conversion: {e}")
        print("Tip: If you run into schema issues, you may need to add ALL_VARCHAR=TRUE to read_csv_auto.")
    finally:
        con.close()

if __name__ == "__main__":
    # Use raw string for Windows file paths
    csv_path = r"C:\Users\rubel\Downloads\open_buildings_v3_polygons_your_own_wkt_polygon.csv\open_buildings_v3_polygons_your_own_wkt_polygon.csv"
    parquet_path = "open_buildings_compressed.parquet"
    
    ingest_csv_to_parquet(csv_path, parquet_path)


Starting ingestion process...
Input CSV: C:\Users\rubel\Downloads\open_buildings_v3_polygons_your_own_wkt_polygon.csv\open_buildings_v3_polygons_your_own_wkt_polygon.csv
Output Parquet: open_buildings_compressed.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! Conversion completed in 944.25 seconds.


In [5]:
import duckdb
import time
import os

def run_spatial_clustering(parquet_path, output_gpkg_path):
    print(f"Starting spatial clustering phase...")
    print(f"Input Parquet: {parquet_path}")
    print(f"Output GeoPackage: {output_gpkg_path}")
    
    start_time = time.time()
    
    # Connect and install/load duckdb spatial extension
    con = duckdb.connect('duckdb_spatial_temp.db')
    con.execute("INSTALL spatial;")
    con.execute("LOAD spatial;")
    
    # 1. Transform Coordinates to UTM
    # EPSG:32643 is UTM zone 43N (common for West India / Mumbai Region)
    # 2. Perform ST_ClusterDBSCAN
    # 3. Aggregate groups via ST_ConcaveHull
    
    # The subqueries handle the nested pipeline without exploding memory.
    query = f"""
    -- DuckDB Spatial SQL implementation for clustering and hull generation
    -- Note: DuckDB's native spatial SQL is still evolving. ST_ClusterDBSCAN may have specific compatibility depending on version.
    
    COPY (
        WITH base_points AS (
            SELECT 
                row_number() OVER () AS id,
                area_in_meters as building_area,
                -- Create geometry point and transform from standard WGS84 to metric UTM Zone 43N
                ST_Transform(ST_Point(longitude, latitude), 'EPSG:4326', 'EPSG:32643') as geom_utm
            FROM read_parquet('{parquet_path}')
            -- Ensure no null coordinates block the geometry creation
            WHERE latitude IS NOT NULL AND longitude IS NOT NULL
        ),
        clustered_points AS (
            SELECT 
                *,
                -- Provide the DBSCAN cluster ID. eps=20 (meters in UTM), min_points=5
                ST_ClusterDBSCAN(geom_utm, 20, 5) OVER() AS cluster_id
            FROM base_points
        ),
        -- Filter out noise
        filtered_clusters AS (
            SELECT * FROM clustered_points WHERE cluster_id IS NOT NULL
        ),
        -- Aggregate stats and build hull per cluster
        aggregate_hulls AS (
            SELECT 
                cluster_id,
                COUNT(*) as building_count,
                SUM(building_area) as total_building_area,
                -- Generate a concave hull for tight cluster bounding
                ST_ConcaveHull(ST_Collect(geom_utm), 0.99) as geom_hull_utm
            FROM filtered_clusters
            GROUP BY cluster_id
        )
        -- Finally, project back to EPSG:4326 for final output
        SELECT 
            cluster_id, 
            building_count, 
            total_building_area, 
            ST_Transform(geom_hull_utm, 'EPSG:32643', 'EPSG:4326') as geom
        FROM aggregate_hulls
    ) TO '{output_gpkg_path}' (FORMAT GDAL, DRIVER 'GPKG');
    """
    
    try:
        print("Executing DuckDB memory-optimized spatial queries... (This may take a while for 35GB of source data)")
        con.execute(query)
        end_time = time.time()
        print(f"Success! Process completed in {(end_time - start_time) / 60:.2f} minutes.")
    except Exception as e:
        print(f"Error during DuckDB clustering: {e}")
        print("Note: If 'ST_ClusterDBSCAN' is unsupported in your current DuckDB version, we will pivot to the high-performance Python cuML/SciKit fallback approach as discussed.")
    finally:
        con.close()

if __name__ == "__main__":
    parquet_in = "open_buildings_compressed.parquet"
    gpkg_out = r"C:\Users\rubel\Downloads\clustered_settlements_epsg4326.gpkg"
    
    run_spatial_clustering(parquet_in, gpkg_out)


Starting spatial clustering phase...
Input Parquet: open_buildings_compressed.parquet
Output GeoPackage: C:\Users\rubel\Downloads\clustered_settlements_epsg4326.gpkg
Executing DuckDB memory-optimized spatial queries... (This may take a while for 35GB of source data)
Error during DuckDB clustering: Catalog Error: Aggregate Function with name st_clusterdbscan does not exist!
Did you mean "st_setcrs"?

LINE 20:                 ST_ClusterDBSCAN(geom_utm, 20, 5) OVER() AS cluster_id
                         ^
Note: If 'ST_ClusterDBSCAN' is unsupported in your current DuckDB version, we will pivot to the high-performance Python cuML/SciKit fallback approach as discussed.


In [6]:
import duckdb
import time
import pandas as pd
from sklearn.cluster import DBSCAN
import shapely.geometry
import geopandas as gpd
from pyproj import Transformer
import warnings

warnings.filterwarnings('ignore')

def run_spatial_clustering_fallback(parquet_path, output_gpkg_path):
    print(f"Starting Python-fallback spatial clustering phase...")
    print(f"Input Parquet: {parquet_path}")
    print(f"Output GeoPackage: {output_gpkg_path}")
    
    start_time = time.time()
    
    # Connect to DuckDB
    con = duckdb.connect('duckdb_spatial_temp.db')
    
    # We will partition the data using a coarse spatial grid (e.g., 1 degree x 1 degree)
    # This prevents Out-Of-Memory errors when applying DBSCAN to 35GB of points.
    # Note: For strict edge-case catching between 1-degree boundaries, a buffer overlap could be 
    # implemented, but simplified grid chunking is the standard fallback.
    
    print("Discovering spatial extents and creating grid...")
    grid_query = f"""
        SELECT 
            CAST(FLOOR(longitude) AS INT) as grid_lon,
            CAST(FLOOR(latitude) AS INT) as grid_lat,
            COUNT(*) as pt_count
        FROM read_parquet('{parquet_path}')
        WHERE latitude IS NOT NULL AND longitude IS NOT NULL
        GROUP BY 1, 2
    """
    grids = con.execute(grid_query).df()
    print(f"Found {len(grids)} spatial grid chunks to process.")
    
    # Setup projection transformer (EPSG:4326 to UTM Zone 43N EPSG:32643) for metric distances
    transformer_to_utm = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True)
    transformer_to_wgs84 = Transformer.from_crs("EPSG:32643", "EPSG:4326", always_xy=True)
    
    all_hulls = []
    global_cluster_offset = 0
    
    for idx, row in grids.iterrows():
        g_lon = row['grid_lon']
        g_lat = row['grid_lat']
        pt_count = row['pt_count']
        
        if pt_count < 5:
            continue # Min points for DBSCAN is 5
            
        print(f"Processing chunk {idx+1}/{len(grids)}: Lon {g_lon}, Lat {g_lat} ({pt_count} points)...")
        
        # Extract chunk
        chunk_query = f"""
            SELECT 
                longitude, 
                latitude,
                area_in_meters as building_area
            FROM read_parquet('{parquet_path}')
            WHERE CAST(FLOOR(longitude) AS INT) = {g_lon}
              AND CAST(FLOOR(latitude) AS INT) = {g_lat}
        """
        df_chunk = con.execute(chunk_query).df()
        
        if df_chunk.empty: continue
            
        # Project to UTM for metric clustering
        df_chunk['x_utm'], df_chunk['y_utm'] = transformer_to_utm.transform(
            df_chunk['longitude'].values, 
            df_chunk['latitude'].values
        )
        
        # Run DBSCAN (eps=20 meters, min_samples=5)
        # Using n_jobs=-1 to use all CPU cores (Ryzen 16 threads)
        coords = df_chunk[['x_utm', 'y_utm']].values
        db = DBSCAN(eps=20, min_samples=5, n_jobs=-1).fit(coords)
        
        df_chunk['cluster_id'] = db.labels_
        
        # Filter noise
        df_valid = df_chunk[df_chunk['cluster_id'] != -1]
        
        if df_valid.empty: continue
            
        # Group by cluster to calculate stats and hulls
        for c_id, group in df_valid.groupby('cluster_id'):
            building_count = len(group)
            total_area = group['building_area'].sum()
            
            # Create multipoint from UTM coordinates
            points = [shapely.geometry.Point(pt) for pt in group[['x_utm', 'y_utm']].values]
            multipoint = shapely.geometry.MultiPoint(points)
            
            # Generate concave hull (alpha shape). If < 3 points, it falls back to convex hull or line
            try:
                # Using convex hull for simplicity and speed. 
                # Concave hull in shapely is concave_hull(geom, ratio=0.99) in newest versions
                hull_utm = multipoint.concave_hull(ratio=0.1) if hasattr(multipoint, 'concave_hull') else multipoint.convex_hull
            except:
                hull_utm = multipoint.convex_hull
                
            # Transform hull back to EPSG:4326
            hull_wgs84 = shapely.ops.transform(transformer_to_wgs84.transform, hull_utm)
            
            all_hulls.append({
                'cluster_id': global_cluster_offset,
                'building_count': building_count,
                'total_building_area': total_area,
                'geometry': hull_wgs84
            })
            global_cluster_offset += 1

    # Save to GeoPackage
    if all_hulls:
        print(f"Clustering complete. Generated {len(all_hulls)} settlements. Saving to GeoPackage...")
        gdf_out = gpd.GeoDataFrame(all_hulls, crs="EPSG:4326")
        gdf_out.to_file(output_gpkg_path, driver="GPKG")
        print("Success! Fallback method completed successfully.")
    else:
        print("No clusters detected.")
        
    con.close()
    
    end_time = time.time()
    print(f"Total processing time: {(end_time - start_time) / 60:.2f} minutes.")

if __name__ == "__main__":
    parquet_in = "open_buildings_compressed.parquet"
    # Using the output path specified in your logs
    gpkg_out = r"C:\Users\rubel\Downloads\clustered_settlements_epsg4326.gpkg"
    
    run_spatial_clustering_fallback(parquet_in, gpkg_out)


Starting Python-fallback spatial clustering phase...
Input Parquet: open_buildings_compressed.parquet
Output GeoPackage: C:\Users\rubel\Downloads\clustered_settlements_epsg4326.gpkg
Discovering spatial extents and creating grid...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Found 155 spatial grid chunks to process.
Processing chunk 1/155: Lon 96, Lat 27 (205216 points)...
Processing chunk 2/155: Lon 90, Lat 27 (47966 points)...
Processing chunk 3/155: Lon 71, Lat 33 (784653 points)...
Processing chunk 4/155: Lon 79, Lat 28 (3523113 points)...
Processing chunk 5/155: Lon 74, Lat 32 (4292906 points)...
Processing chunk 6/155: Lon 74, Lat 31 (2904377 points)...
Processing chunk 7/155: Lon 88, Lat 28 (3095 points)...
Processing chunk 8/155: Lon 86, Lat 25 (729264 points)...
Processing chunk 9/155: Lon 85, Lat 25 (299816 points)...
Processing chunk 10/155: Lon 75, Lat 36 (2052 points)...
Processing chunk 11/155: Lon 81, Lat 31 (456 points)...
Processing chunk 12/155: Lon 98, Lat 28 (350 points)...
Processing chunk 13/155: Lon 77, Lat 28 (945554 points)...
Processing chunk 14/155: Lon 76, Lat 29 (935029 points)...
Processing chunk 15/155: Lon 84, Lat 27 (1649633 points)...
Processing chunk 16/155: Lon 82, Lat 28 (614835 points)...
Processing chunk 17/155: Lon 8

In [2]:
import geopandas as gpd
aoi = gpd.read_file(r"D:\Rubel\M.Tech\MTP\Phase 2\Aoi\aoi1.shp")

In [3]:
import os
import glob
import geopandas as gpd
import pandas as pd

def process_osm_roads(data_folder, aoi_path, output_gpkg):
    print("Loading AOI...")
    aoi = gpd.read_file(aoi_path)
    target_crs = aoi.crs
    print(f"AOI CRS: {target_crs}")

    # Find all GeoPackage files in the folder
    gpkg_files = glob.glob(os.path.join(data_folder, '*.gpkg'))
    print(f"Found {len(gpkg_files)} GeoPackage files.")

    clipped_gdfs = []

    for gpkg in gpkg_files:
        filename = os.path.basename(gpkg)
        print(f"\nProcessing {filename}...")
        try:
            # Read the specific layer
            # Using mask=aoi helps filtering at load time if their CRSes match
            gdf = gpd.read_file(gpkg, layer='gis_osm_roads_free')
            
            if gdf.empty:
                print("  Layer 'gis_osm_roads_free' is empty.")
                continue

            # Ensure CRS perfectly aligns with AOI to perform the clip
            if gdf.crs != target_crs:
                print(f"  Reprojecting from {gdf.crs} to {target_crs}...")
                gdf = gdf.to_crs(target_crs)

            # Clip individually before merging to save massive amounts of RAM
            print("  Clipping to AOI...")
            clipped = gpd.clip(gdf, aoi)

            if not clipped.empty:
                # Keep only LineStrings and MultiLineStrings
                clipped = clipped[clipped.geometry.type.isin(["LineString", "MultiLineString"])]
                clipped_gdfs.append(clipped)
                print(f"  Added {len(clipped)} road segments from this file.")
            else:
                print("  No roads intersect with the AOI.")

        except Exception as e:
            print(f"  Error processing {filename}. Ensure it has the 'gis_osm_roads_free' layer.")
            print(f"  Details: {e}")

    # Merge everything into a single GeoDataFrame
    if clipped_gdfs:
        print("\nMerging all extracted and clipped roads...")
        final_merged = gpd.GeoDataFrame(pd.concat(clipped_gdfs, ignore_index=True), crs=target_crs)
        
        print(f"Final merged dataset contains {len(final_merged)} road segments.")
        print(f"Saving to {output_gpkg}...")
        final_merged.to_file(output_gpkg, driver="GPKG")
        print("Success! Process completed.")
    else:
        print("\nNo roads extracted. Check your AOI boundaries or input GeoPackages.")

if __name__ == "__main__":
    folder_path = r"C:\Users\rubel\Downloads\osm\data"
    aoi_shapefile = r"D:\Rubel\M.Tech\MTP\Phase 2\Aoi\aoi1.shp"
    output_file = r"C:\Users\rubel\Downloads\osm\data\final_merged_clipped_roads.gpkg"

    process_osm_roads(folder_path, aoi_shapefile, output_file)


Loading AOI...
AOI CRS: EPSG:4326
Found 10 GeoPackage files.

Processing afghanistan.gpkg...
  Reprojecting from GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Longitude",EAST],AXIS["Latitude",NORTH],AUTHORITY["OGC","CRS84"]] to EPSG:4326...
  Clipping to AOI...
  Added 47986 road segments from this file.

Processing bhutan.gpkg...
  Reprojecting from GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Longitude",EAST],AXIS["Latitude",NORTH],AUTHORITY["OGC","CRS84"]] to EPSG:4326...
  Clipping to AOI...
  Added 30544 road segments from this file.

Processing central-zone.gpkg...
  Reprojecting from GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG

In [5]:
roads = gpd.read_file(r"C:\Users\rubel\Downloads\osm\data\final_merged_clipped_roads.gpkg")
roads.columns

Index(['osm_id', 'code', 'fclass', 'name', 'ref', 'oneway', 'maxspeed',
       'layer', 'bridge', 'tunnel', 'geometry'],
      dtype='str')

In [6]:
roads['code'].unique()

array([5142, 5121, 5115, 5154, 5122, 5112, 5114, 5132, 5123, 5141, 5113,
       5153, 5133, 5135, 5134, 5143, 5144, 5145, 5146, 5199, 5147, 5151,
       5155, 5124, 5152, 5125, 5111, 5131], dtype=int32)

In [ ]:
# Dissolve by class/code to group road types
# Note: This groups everything with the same code. 
# While it creates a single MultiLineString per class, the pieces inside might not be topologically continuous (connected).
grouped_roads = roads.dissolve(by='code', as_index=False)


In [7]:
# If you specifically want to fuse individual segments that actually touch into long continuous linestrings:
import geopandas as gpd
import shapely

print("Fusing topologically continuous segments...")
# First, explode any MultiLineStrings into individual LineStrings to prevent geometry sequence errors
roads_exploded = roads.explode(index_parts=False)

# polygonize/linemerge will attempt to connect touching endpoints of lines 
# Using .tolist() instead of .values to prevent numpy array truth value ambiguity errors
fused_lines = shapely.ops.linemerge(roads_exploded.geometry.tolist())

# Linemerge returns a MultiLineString if there are disconnected graphes, or LineString if everything connects
if isinstance(fused_lines, shapely.geometry.LineString):
    fused_gseries = gpd.GeoSeries([fused_lines])
else:
    fused_gseries = gpd.GeoSeries(list(fused_lines.geoms))

# Create a new simplified dataframe of topologically continuous roads
continuous_roads = gpd.GeoDataFrame(geometry=fused_gseries, crs=roads.crs)
print(f"Original segments: {len(roads)} -> Fused segments: {len(continuous_roads)}")

Fusing topologically continuous segments...
Original segments: 845201 -> Fused segments: 621425


In [8]:
continuous_roads.to_file(r"C:\Users\rubel\Downloads\osm\data\final_fused_continuous_roads.gpkg", driver="GPKG")

In [16]:
roads['oneway'].unique()

array(['B', 'F', 'T'], dtype=object)

In [ ]:
buildings = gpd.read_file(r"C:\Users\rubel\Downloads\osm\data\nepal.gpkg",layer= 'gis_osm_buildings_a_free')


In [1]:
import os
import glob
import geopandas as gpd
import pandas as pd

def process_osm_buildings(data_folder, aoi_path, output_gpkg):
    print("Loading AOI...")
    aoi = gpd.read_file(aoi_path)
    target_crs = aoi.crs
    print(f"AOI CRS: {target_crs}")

    # Find all GeoPackage files in the folder
    gpkg_files = glob.glob(os.path.join(data_folder, '*.gpkg'))
    print(f"Found {len(gpkg_files)} GeoPackage files.")

    clipped_gdfs = []

    for gpkg in gpkg_files:
        filename = os.path.basename(gpkg)
        print(f"\nProcessing {filename}...")
        try:
            # Read the specific layer
            # Using mask=aoi helps filtering at load time if their CRSes match
            gdf = gpd.read_file(gpkg, layer='gis_osm_buildings_a_free')
            
            if gdf.empty:
                print("  Layer 'gis_osm_buildings_a_free' is empty.")
                continue

            # Ensure CRS perfectly aligns with AOI to perform the clip
            if gdf.crs != target_crs:
                print(f"  Reprojecting from {gdf.crs} to {target_crs}...")
                gdf = gdf.to_crs(target_crs)

            # Clip individually before merging to save massive amounts of RAM
            print("  Clipping to AOI...")
            clipped = gpd.clip(gdf, aoi)

            if not clipped.empty:
                # Keep only Polygon and MultiPolygon geometries
                clipped = clipped[clipped.geometry.type.isin(["Polygon", "MultiPolygon"])]
                clipped_gdfs.append(clipped)
                print(f"  Added {len(clipped)} building polygons from this file.")
            else:
                print("  No buildings intersect with the AOI.")

        except Exception as e:
            print(f"  Error processing {filename}. Ensure it has the 'gis_osm_buildings_a_free' layer.")
            print(f"  Details: {e}")

    # Merge everything into a single GeoDataFrame
    if clipped_gdfs:
        print("\nMerging all extracted and clipped buildings...")
        final_merged = gpd.GeoDataFrame(pd.concat(clipped_gdfs, ignore_index=True), crs=target_crs)
        
        print(f"Final merged dataset contains {len(final_merged)} building polygons.")
        print(f"Saving to {output_gpkg}...")
        final_merged.to_file(output_gpkg, driver="GPKG")
        print("Success! Process completed.")
    else:
        print("\nNo buildings extracted. Check your AOI boundaries or input GeoPackages.")

if __name__ == "__main__":
    folder_path = r"C:\Users\rubel\Downloads\osm\data"
    aoi_shapefile = r"D:\Rubel\M.Tech\MTP\Phase 2\Aoi\aoi1.shp"
    output_file = r"C:\Users\rubel\Downloads\osm\data\final_merged_clipped_buildings.gpkg"

    process_osm_buildings(folder_path, aoi_shapefile, output_file)


Loading AOI...
AOI CRS: EPSG:4326
Found 12 GeoPackage files.

Processing afghanistan.gpkg...
  Reprojecting from GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Longitude",EAST],AXIS["Latitude",NORTH],AUTHORITY["OGC","CRS84"]] to EPSG:4326...
  Clipping to AOI...
  Added 130488 building polygons from this file.

Processing bhutan.gpkg...
  Reprojecting from GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Longitude",EAST],AXIS["Latitude",NORTH],AUTHORITY["OGC","CRS84"]] to EPSG:4326...
  Clipping to AOI...
  Added 187777 building polygons from this file.

Processing central-zone.gpkg...
  Reprojecting from GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHO

In [2]:
import os
from pathlib import Path

import fiona
import geopandas as gpd
import pandas as pd


def _pick_layer(gpkg_path, preferred_keywords=None):
    layers = fiona.listlayers(gpkg_path)
    if not layers:
        raise ValueError(f'No layers found in {gpkg_path}')

    if len(layers) == 1:
        return layers[0]

    preferred_keywords = preferred_keywords or []
    for keyword in preferred_keywords:
        for layer_name in layers:
            if keyword.lower() in layer_name.lower():
                return layer_name

    return layers[0]


def filter_buildings_by_glacial_lake_buffer(
    buildings_gpkg,
    lakes_gpkg,
    output_gpkg,
    buffer_km=25,
):
    print('Loading building and glacial lake layers...')

    buildings_layer = _pick_layer(buildings_gpkg, preferred_keywords=['building', 'buildings'])
    lakes_layer = _pick_layer(lakes_gpkg, preferred_keywords=['lake', 'glacial'])

    buildings = gpd.read_file(buildings_gpkg, layer=buildings_layer)
    lakes = gpd.read_file(lakes_gpkg, layer=lakes_layer)

    if buildings.empty:
        raise ValueError(f'No building features found in {buildings_gpkg}')
    if lakes.empty:
        raise ValueError(f'No lake features found in {lakes_gpkg}')
    if buildings.crs is None:
        raise ValueError('Buildings layer has no CRS defined')
    if lakes.crs is None:
        raise ValueError('Lake layer has no CRS defined')

    work_crs = lakes.crs
    if work_crs.is_geographic:
        work_crs = lakes.estimate_utm_crs()

    print(f'Using working CRS: {work_crs}')
    print(f'Buildings layer: {buildings_layer}')
    print(f'Lake layer: {lakes_layer}')

    buildings_m = buildings.to_crs(work_crs)
    lakes_m = lakes.to_crs(work_crs)
    buffer_m = buffer_km * 1000

    lake_name_col = next(
        (col for col in ['lake_name', 'name', 'lake', 'glacial_lake', 'feature_name'] if col in lakes_m.columns),
        None,
    )

    results = []
    total_lakes = len(lakes_m)

    print(f'Processing {total_lakes} glacial lakes with a {buffer_km} km buffer...')

    for lake_index, lake_row in lakes_m.iterrows():
        lake_geom = lake_row.geometry
        if lake_geom is None or lake_geom.is_empty:
            continue

        lake_buffer = lake_geom.buffer(buffer_m)

        candidate_idx = list(buildings_m.sindex.query(lake_buffer, predicate='intersects'))
        if not candidate_idx:
            continue

        candidate_buildings = buildings_m.iloc[candidate_idx].copy()
        candidate_buildings = candidate_buildings[candidate_buildings.intersects(lake_buffer)]
        if candidate_buildings.empty:
            continue

        candidate_buildings['glacial_lake_id'] = lake_index
        if lake_name_col is not None:
            candidate_buildings['glacial_lake_name'] = lake_row[lake_name_col]
        candidate_buildings['buffer_km'] = buffer_km

        results.append(candidate_buildings)
        print(f'  Lake {lake_index + 1}/{total_lakes}: kept {len(candidate_buildings)} buildings')

    if not results:
        print('No buildings found inside the requested lake buffer.')
        return None

    filtered_buildings = gpd.GeoDataFrame(
        pd.concat(results, ignore_index=True),
        crs=work_crs,
    )

    if buildings.crs != work_crs:
        filtered_buildings = filtered_buildings.to_crs(buildings.crs)

    output_path = Path(output_gpkg)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    filtered_buildings.to_file(output_path, driver='GPKG')

    print(f'Saved {len(filtered_buildings)} buildings to {output_path}')
    return filtered_buildings


buildings_gpkg = r'C:\Users\rubel\Downloads\osm\data\final_merged_clipped_buildings.gpkg'
lakes_gpkg = r'D:\Rubel\M.Tech\MTP\Phase 2\model_data\Final data\stage2_dataset_15.gpkg'
output_gpkg = r'C:\Users\rubel\Downloads\osm\data\buildings_within_glacial_lake_buffer_25km.gpkg'

filtered_buildings = filter_buildings_by_glacial_lake_buffer(
    buildings_gpkg=buildings_gpkg,
    lakes_gpkg=lakes_gpkg,
    output_gpkg=output_gpkg,
    buffer_km=25,
)


Cannot find header.dxf (GDAL_DATA is not defined)


Loading building and glacial lake layers...
Using working CRS: PROJCS["Asia_North_Albers_Equal_Area_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",30],PARAMETER["longitude_of_center",95],PARAMETER["standard_parallel_1",15],PARAMETER["standard_parallel_2",65],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102025"]]
Buildings layer: final_merged_clipped_buildings
Lake layer: stage2_dataset_15
Processing 7088 glacial lakes with a 25 km buffer...
  Lake 2/7088: kept 205 buildings
  Lake 3/7088: kept 205 buildings
  Lake 4/7088: kept 45 buildings
  Lake 5/7088: kept 36 buildings
  Lake 6/7088: kep

In [ ]:
import geopandas as gpd
buildings = gpd.read_file(r"C:\Users\rubel\Downloads\osm\data\northern-zone.gpkg", layer='gis_osm_buildings_a_free')
buildings.columns

NameError: name 'gpd' is not defined

In [8]:
buildings['type'].unique().tolist()

['',
 'train_station',
 'industrial',
 'office',
 'hotel',
 'commercial',
 'parking',
 'roof',
 'government',
 'apartments',
 'church',
 'temple',
 'Stadium',
 'transportation',
 'cathedral',
 'hut',
 'tomb',
 'school',
 'ruins',
 'gurdwara',
 'retail',
 'mosque',
 'college',
 'gurudwara',
 'residential',
 'public',
 'Offical',
 'police',
 'stadium',
 'h',
 'clinic',
 'dormitory',
 'grandstand',
 'house',
 'castle',
 'no',
 'bunker',
 'hangar',
 'hospital',
 'university',
 'kindergarten',
 'manufacture',
 'terrace',
 'garage',
 'service',
 'construction',
 'Community_Hall',
 'councourse',
 'handloom',
 'administrative',
 'houseboat',
 'supermarket',
 'auditorium',
 'wall',
 'C_Block',
 'detached',
 'semidetached_house',
 'shed',
 'fire_station',
 'sports_hall',
 'toilets',
 'storage_tank',
 'place_of_worship',
 'car_detailing_studio',
 'bungalow',
 'Charan_Singh_Complex',
 'cafe',
 'School',
 'warehouse',
 'National Bureau of P',
 'hostel',
 'pavilion',
 'tent',
 'Govt_School',
 'kiosk

In [9]:
buildings['code'].unique()

array([1500], dtype=int32)